In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine

In [2]:
# PATHS

RAW = r"G:\bluestock_mf_capstone\data\raw"
OUT = r"G:\bluestock_mf_capstone\data\processed"
DB_DIR = r"G:\bluestock_mf_capstone\data\db"

os.makedirs(OUT, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

DB = os.path.join(DB_DIR, "bluestock_mf.db")

In [3]:
# LOAD CSV FILES

fund = pd.read_csv(f"{RAW}\\01_fund_master.csv")
nav = pd.read_csv(f"{RAW}\\02_nav_history.csv")
aum = pd.read_csv(f"{RAW}\\03_aum_by_fund_house.csv")
sip = pd.read_csv(f"{RAW}\\04_monthly_sip_inflows.csv")
cat = pd.read_csv(f"{RAW}\\05_category_inflows.csv")
folio = pd.read_csv(f"{RAW}\\06_industry_folio_count.csv")
perf = pd.read_csv(f"{RAW}\\07_scheme_performance.csv")
tx = pd.read_csv(f"{RAW}\\08_investor_transactions.csv")
ph = pd.read_csv(f"{RAW}\\09_portfolio_holdings.csv")
bi = pd.read_csv(f"{RAW}\\10_benchmark_indices.csv")

In [4]:
# STANDARDIZE COLUMN NAMES

all_dfs = [fund, nav, aum, sip, cat, folio, perf, tx, ph, bi]

for df in all_dfs:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [5]:
# NAV HISTORY CLEANING

nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav = nav.drop_duplicates()

nav["nav"] = pd.to_numeric(
    nav["nav"],
    errors="coerce"
)

nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
    .ffill()
)

nav = nav[nav["nav"] > 0]

In [6]:
# INVESTOR TRANSACTIONS CLEANING

tx["transaction_date"] = pd.to_datetime(
    tx["transaction_date"],
    errors="coerce"
)

tx["transaction_type"] = (
    tx["transaction_type"]
    .astype(str)
    .str.lower()
    .replace({
        "sip": "SIP",
        "sip payment": "SIP",
        "lump sum": "Lumpsum",
        "lumpsum": "Lumpsum",
        "redeem": "Redemption"
    })
)

tx["amount_inr"] = pd.to_numeric(
    tx["amount_inr"],
    errors="coerce"
)

tx = tx[
    tx["amount_inr"] > 0
]

In [7]:
# KYC STATUS VALIDATION

print("\n========== KYC STATUS CHECK ==========")

if "kyc_status" in tx.columns:

    tx["kyc_status"] = (
        tx["kyc_status"]
        .astype(str)
        .str.strip()
    )

    print(tx["kyc_status"].value_counts())

    valid_kyc = [
        "Verified",
        "Pending",
        "Rejected"
    ]

    invalid_kyc = tx[
        ~tx["kyc_status"].isin(valid_kyc)
    ]

    print(
        "\nInvalid KYC Records:",
        len(invalid_kyc)
    )


========== KYC STATUS CHECK ==========
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Invalid KYC Records: 0


In [8]:
# SCHEME PERFORMANCE CLEANING

perf_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "expense_ratio_pct"
]

for col in perf_cols:

    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

perf = perf[
    (perf["expense_ratio_pct"] >= 0.1)
    &
    (perf["expense_ratio_pct"] <= 2.5)
]

In [9]:
# PERFORMANCE ANOMALY CHECK

print("\n========== PERFORMANCE ANOMALY CHECK ==========")

anomalies = perf[
    (perf["return_1yr_pct"] > 100)
    |
    (perf["return_1yr_pct"] < -100)
    |
    (perf["return_3yr_pct"] > 100)
    |
    (perf["return_3yr_pct"] < -100)
    |
    (perf["return_5yr_pct"] > 100)
    |
    (perf["return_5yr_pct"] < -100)
]

print(
    "Anomalous Records Found:",
    len(anomalies)
)

if len(anomalies) > 0:

    print(
        anomalies[
            [
                "scheme_name",
                "return_1yr_pct",
                "return_3yr_pct",
                "return_5yr_pct"
            ]
        ].head()
    )


========== PERFORMANCE ANOMALY CHECK ==========
Anomalous Records Found: 0


In [10]:
# SAVE CLEAN FILES

nav.to_csv(
    f"{OUT}\\nav_clean.csv",
    index=False
)

tx.to_csv(
    f"{OUT}\\transactions_clean.csv",
    index=False
)

perf.to_csv(
    f"{OUT}\\performance_clean.csv",
    index=False
)

fund.to_csv(
    f"{OUT}\\fund_clean.csv",
    index=False
)

aum.to_csv(
    f"{OUT}\\aum_clean.csv",
    index=False
)

sip.to_csv(
    f"{OUT}\\sip_clean.csv",
    index=False
)

cat.to_csv(
    f"{OUT}\\category_clean.csv",
    index=False
)

folio.to_csv(
    f"{OUT}\\folio_clean.csv",
    index=False
)

ph.to_csv(
    f"{OUT}\\holdings_clean.csv",
    index=False
)

bi.to_csv(
    f"{OUT}\\benchmark_clean.csv",
    index=False
)


# SQLITE DATABASE LOAD

engine = create_engine(
    f"sqlite:///{DB}"
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

tx.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

fund.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

aum.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)

sip.to_sql(
    "fact_sip",
    engine,
    if_exists="replace",
    index=False
)

cat.to_sql(
    "fact_category",
    engine,
    if_exists="replace",
    index=False
)

folio.to_sql(
    "fact_folio",
    engine,
    if_exists="replace",
    index=False
)

ph.to_sql(
    "fact_holdings",
    engine,
    if_exists="replace",
    index=False
)

bi.to_sql(
    "fact_benchmark",
    engine,
    if_exists="replace",
    index=False
)

8050

In [11]:
# DATA SUMMARY

print("\n========== DATA SUMMARY ==========")

print("Fund Master:", fund.shape)
print("NAV History:", nav.shape)
print("AUM:", aum.shape)
print("SIP:", sip.shape)
print("Category:", cat.shape)
print("Folio:", folio.shape)
print("Performance:", perf.shape)
print("Transactions:", tx.shape)
print("Holdings:", ph.shape)
print("Benchmark:", bi.shape)

print("\nNAV Columns:")
print(nav.columns.tolist())

print("\nTransaction Columns:")
print(tx.columns.tolist())

print("\nPerformance Columns:")
print(perf.columns.tolist())


========== DATA SUMMARY ==========
Fund Master: (40, 15)
NAV History: (46000, 3)
AUM: (90, 5)
SIP: (48, 6)
Category: (144, 3)
Folio: (21, 6)
Performance: (40, 19)
Transactions: (32778, 13)
Holdings: (322, 8)
Benchmark: (8050, 3)

NAV Columns:
['amfi_code', 'date', 'nav']

Transaction Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Performance Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [12]:
# SUCCESS MESSAGE

print("\nDAY 2 PIPELINE COMPLETED SUCCESSFULLY ✅")
print(f"Database Created: {DB}")
print(f"Processed Files Saved In: {OUT}")


DAY 2 PIPELINE COMPLETED SUCCESSFULLY ✅
Database Created: G:\bluestock_mf_capstone\data\db\bluestock_mf.db
Processed Files Saved In: G:\bluestock_mf_capstone\data\processed
